# ModernBERT Clustered SPLADE Analysis

This notebook analyzes the learned clusters and activation patterns of the ModernBERT checkpoint.

## Objectives:
1. Map each of the 8,000 dimensions to their corresponding tokens.
2. Analyze activation frequency across a sample of 1000 articles.
3. Visualize key dimensions and tokens for sample articles.

In [ ]:
import json
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Load data
mapping_path = 'cluster_mapping.json'
freq_path = 'results/frequencies.json'
samples_path = 'results/sample_activations.json'

with open(mapping_path, 'r') as f:
    mapping = json.load(f)

with open(freq_path, 'r') as f:
    frequencies = json.load(f)

with open(samples_path, 'r') as f:
    samples = json.load(f)

print(f"Loaded {len(mapping)} clusters and frequency data for {len(frequencies)} active clusters.")

## 1. Top Frequent Clusters
Which clusters are most commonly activated across the 1000 articles?

In [ ]:
# Convert frequencies to a sorted list
sorted_freq = sorted(frequencies.items(), key=lambda x: x[1], reverse=True)

top_20 = sorted_freq[:20]
data = []
for cid, count in top_20:
    tokens = mapping.get(cid, [])
    data.append({
        'Cluster ID': cid,
        'Frequency': count,
        'Tokens': ", ".join(tokens[:10]) + ("..." if len(tokens) > 10 else "")
    })

df_top = pd.DataFrame(data)
display(df_top)

# Visualization
plt.figure(figsize=(12, 6))
sns.barplot(x='Frequency', y='Cluster ID', data=df_top, palette='viridis')
plt.title('Top 20 Most Frequent Clusters (Across 1000 Articles)')
plt.show()

## 2. Token Analysis per Dimension
Let's look at the mapping for a few interesting clusters.

In [ ]:
def inspect_cluster(cid):
    cid = str(cid)
    tokens = mapping.get(cid, [])
    print(f"### Cluster {cid} ###")
    print(f"Tokens: {tokens}")
    print(f"Frequency: {frequencies.get(cid, 0)}")
    print("-" * 20)

# Inspect top 5
for cid, _ in top_20[:5]:
    inspect_cluster(cid)

## 3. Sample Article Analysis
For a few articles, which dimensions are highly activated?

In [ ]:
for sample in samples[:3]:
    print(f"\n--- Article {sample['article_index']} ---")
    print(f"Text: {sample['text_snippet']}")
    print("Top Dimensions:")
    for act in sample['top_activations'][:5]:
        cid = str(act['cluster_id'])
        tokens = mapping.get(cid, [])
        print(f"  [Dim {cid}] Score: {act['score']:.4f} | Tokens: {tokens[:5]}")

## Replication Guide
To replicate this analysis from scratch:
1. Run `cluster_analysis.py` to generate `cluster_mapping.json` using the base model weights.
2. Run `activation_analysis.py` to generate activation data on your chosen dataset (e.g., `amazon_triplets.jsonl`).
3. Use this notebook to visualize and inspect the results.